In [ ]:
#| default_exp train_flow

# Flow Matching Generative Model

Trains a flow matching model on pre-encoded (optionally PCA-reduced) embeddings.
Source distribution is N(0,I); target is the embedding distribution.
Uses RK4 integration and optional time warping at inference.

In [ ]:
#| export
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from midi_rae.data import EmbeddingDataset

In [ ]:
#| export
class VelocityNet(nn.Module):
    """MLP velocity field for flow matching.  Input: [x, t], output: dx/dt.
    Hidden layers use residual (skip) connections."""
    def __init__(self, input_dim, h_dim=256, n_layers=3):
        super().__init__()
        self.fc_in  = nn.Linear(input_dim + 1, h_dim)
        self.hidden = nn.ModuleList([nn.Linear(h_dim, h_dim) for _ in range(n_layers - 1)])
        self.fc_out = nn.Linear(h_dim, input_dim)

    def forward(self, x, t):
        if t.dim() == 0: t = t.expand(x.size(0), 1)
        elif t.dim() == 1: t = t.unsqueeze(1)
        t = t.expand(x.size(0), 1)
        h = F.gelu(self.fc_in(torch.cat([x, t], dim=1)))
        for layer in self.hidden:
            h = F.gelu(layer(h)) + h  # residual connection
        return self.fc_out(h)

In [ ]:
#| export
class PerLevelFlowModel(nn.Module):
    """One VelocityNet per embedding level; each level's slice is routed to its own net.
    Has the same forward(x, t) interface as VelocityNet, so all training/eval code works unchanged.
    level_dims: list of ints, e.g. [16, 16, 16, 16, 16, 8] from dataset.level_dims
    """
    def __init__(self, level_dims, h_dim=256, n_layers=4):
        super().__init__()
        self.level_dims = level_dims
        self.nets = nn.ModuleList([VelocityNet(d, h_dim, n_layers) for d in level_dims])

    def forward(self, x, t):
        outs, offset = [], 0
        for net, d in zip(self.nets, self.level_dims):
            outs.append(net(x[:, offset:offset+d], t))
            offset += d
        return torch.cat(outs, dim=1)

In [ ]:
#| export
def warp_time(t, s=0.5):
    """Parametric time warping (Scott H. Hawley, 'Flow With What You Know', ICLR 2025).
    s=1 → linear; s<1 → slower near middle; s=1.5 ≈ cosine schedule.
    Works on scalar, 1-D or 2-D tensors."""
    return 4*(1-s)*t**3 + 6*(s-1)*t**2 + (3-2*s)*t

In [ ]:
#| export
@torch.no_grad()
def rk4_step(model, y, t, dt):
    """4th-order Runge-Kutta step for the learned velocity field."""
    t_  = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    k1 = model(y,             t_)
    k2 = model(y + dt*k1/2,   t_ + dt/2)
    k3 = model(y + dt*k2/2,   t_ + dt/2)
    k4 = model(y + dt*k3,     t_ + dt)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

@torch.no_grad()
def euler_step(model, y, t, dt):
    t_ = torch.full((y.size(0), 1), t, device=y.device, dtype=y.dtype)
    return y + model(y, t_) * dt

In [ ]:
#| export
def sample_source(shape, device='cpu', source_df=None, source_scales=None, level_dims=None):
    """Sample from source distribution with optional per-level Student-t and scaling.
    source_df: scalar df → Student-t for all dims; list → per-level (None/0 = Gaussian, float = Student-t)
    source_scales: list of per-level scale factors applied after sampling
    """
    if isinstance(source_df, (list, tuple)):
        # Per-level: each slice sampled independently
        assert level_dims is not None, "level_dims required for per-level source_df"
        batch = shape[:-1]
        y = torch.empty(*shape, device=device)
        offset = 0
        for df, d in zip(source_df, level_dims):
            sl = (*batch, d)
            if df:
                normal = torch.randn(*sl, device=device)
                gamma  = torch._standard_gamma(torch.full(sl, df/2, device=device)) / (df/2)
                y[..., offset:offset+d] = normal / gamma.sqrt()
            else:
                y[..., offset:offset+d] = torch.randn(*sl, device=device)
            offset += d
    elif source_df:
        # Scalar df → Student-t for all dims
        normal = torch.randn(*shape, device=device)
        gamma  = torch._standard_gamma(torch.full(shape, source_df/2, device=device)) / (source_df/2)
        y = normal / gamma.sqrt()
    else:
        y = torch.randn(*shape, device=device)
    if source_scales is not None and level_dims is not None:
        offset = 0
        for scale, d in zip(source_scales, level_dims):
            y[..., offset:offset+d] *= scale
            offset += d
    return y

In [ ]:
#| export
@torch.no_grad()
def generate_samples(model, n_samples, dim, device='cpu',
                     n_steps=20, step_fn=rk4_step, warp_s=0.5, source_df=None,
                     source_scales=None, level_dims=None):
    """Sample from the flow model: integrate noise → embedding space."""
    y = sample_source((n_samples, dim), device=device, source_df=source_df,
                      source_scales=source_scales, level_dims=level_dims)
    ts = torch.linspace(0, 1, n_steps + 1)
    ts = warp_time(ts, s=warp_s)
    model.eval()
    for i in range(n_steps):
        dt = (ts[i+1] - ts[i]).item()
        y  = step_fn(model, y, ts[i].item(), dt)
    return y

In [ ]:
#| export
def mmd_rbf(x, y, n_sub=2000):
    """Unbiased MMD² with RBF kernel, median bandwidth heuristic.
    x, y: (N, D) tensors. Subsamples to n_sub for speed."""
    if x.size(0) > n_sub: x = x[torch.randperm(x.size(0))[:n_sub]]
    if y.size(0) > n_sub: y = y[torch.randperm(y.size(0))[:n_sub]]
    xy = torch.cat([x, y], dim=0)
    sigma2 = torch.cdist(xy, xy).median().pow(2).clamp(min=1e-6)
    def rbf(a, b): return torch.exp(-torch.cdist(a, b).pow(2) / (2 * sigma2))
    return (rbf(x, x).mean() + rbf(y, y).mean() - 2 * rbf(x, y).mean()).item()


In [ ]:

#| export
def wasserstein_score(x, y, n_projections=200, n_sub=2000):
    """Sliced Wasserstein distance: average 1-D Wasserstein over random projections.
    Falls back gracefully if geomloss is unavailable.
    x, y: (N, D) numpy arrays."""
    try:
        import geomloss
        loss = geomloss.SamplesLoss("sinkhorn", p=2, blur=0.05)
        xt = torch.tensor(x[:n_sub]).float()
        yt = torch.tensor(y[:n_sub]).float()
        return loss(xt, yt).item()
    except ImportError:
        pass
    from scipy.stats import wasserstein_distance
    rng = np.random.default_rng(0)
    D = x.shape[1]
    projs = rng.standard_normal((D, n_projections))
    projs /= np.linalg.norm(projs, axis=0, keepdims=True)
    px, py = x[:n_sub] @ projs, y[:n_sub] @ projs
    return float(np.mean([wasserstein_distance(px[:, i], py[:, i]) for i in range(n_projections)]))


In [ ]:

#| export
@torch.no_grad()
def eval_flow(model, real_embeddings, n_samples=10000, n_steps=20, warp_s=0.5, device='cpu',
              source_df=None, source_scales=None, level_dims=None):
    """Compare distributional statistics of real vs generated embeddings, per level.
    Returns flat dict with keys like 'L0/mmd', 'L0/wasserstein', 'L0/real_std', etc.
    Also returns global 'mmd' and 'wasserstein' for backward compatibility.
    real_embeddings: (N, D) tensor.
    """
    from scipy.stats import skew, kurtosis
    dim = real_embeddings.shape[1]
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float()
    gen  = generate_samples(model, n_samples, dim, device=device,
                            n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                            source_scales=source_scales, level_dims=level_dims).cpu()
    r, g = real.numpy(), gen.numpy()

    metrics = {}
    # Global stats
    metrics['real_mean']  = float(r.mean())
    metrics['real_std']   = float(r.std())
    metrics['real_skew']  = float(skew(r.ravel()))
    metrics['real_kurt']  = float(kurtosis(r.ravel()))
    metrics['gen_mean']   = float(g.mean())
    metrics['gen_std']    = float(g.std())
    metrics['gen_skew']   = float(skew(g.ravel()))
    metrics['gen_kurt']   = float(kurtosis(g.ravel()))
    metrics['mmd']        = mmd_rbf(real, gen)
    metrics['wasserstein'] = wasserstein_score(r, g)

    # Per-level stats
    if level_dims is not None:
        offset = 0
        for i, d in enumerate(level_dims):
            rl = r[:, offset:offset+d]
            gl = g[:, offset:offset+d]
            rt, gt = torch.tensor(rl), torch.tensor(gl)
            metrics[f'L{i}/real_std']    = float(rl.std())
            metrics[f'L{i}/gen_std']     = float(gl.std())
            metrics[f'L{i}/real_kurt']   = float(kurtosis(rl.ravel()))
            metrics[f'L{i}/gen_kurt']    = float(kurtosis(gl.ravel()))
            metrics[f'L{i}/mmd']         = mmd_rbf(rt, gt)
            metrics[f'L{i}/wasserstein'] = wasserstein_score(rl, gl)
            offset += d

    w = max(len(k) for k in metrics)
    for k, v in metrics.items():
        print(f'  {k:{w}s} = {v:.4f}')
    return metrics


In [ ]:

#| export
@torch.no_grad()
def plot_level_histograms(model, real_embeddings, level_dims, n_samples=10000,
                          n_steps=20, warp_s=0.5, device='cpu', n_bins=100, source_df=None,
                          source_scales=None, epoch=None):
    """Return dict of per-level histogram figures {'L0': fig, 'L1': fig, ...}.
    level_dims: list of ints, flattened PCA dims per level e.g. [20, 80, 320, 1280]
    real_embeddings: (N, sum(level_dims)) tensor
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    dim = real_embeddings.shape[1]
    idx = torch.randperm(real_embeddings.size(0))[:n_samples]
    real = real_embeddings[idx].float().numpy()
    gen  = generate_samples(model, n_samples, dim, device=device,
                            n_steps=n_steps, warp_s=warp_s, source_df=source_df,
                            source_scales=source_scales, level_dims=level_dims).cpu().numpy()
    figs = {}
    offset = 0
    for i, d in enumerate(level_dims):
        r = real[:, offset:offset+d].ravel()
        g = gen[:,  offset:offset+d].ravel()
        lim = np.percentile(np.abs(np.concatenate([r, g])), 99)
        bins = np.linspace(-lim, lim, n_bins + 1)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(r, bins=bins, alpha=0.5, color='steelblue', label='real', density=True)
        ax.hist(g, bins=bins, alpha=0.5, color='darkorange', label='gen',  density=True)
        title = f'L{i} ({d}d)'
        if epoch is not None: title += f' — Epoch {epoch}'
        ax.set_title(title)
        ax.set_xlabel('value')
        ax.legend(fontsize=8)
        plt.tight_layout()
        figs[f'L{i}'] = fig
        offset += d
    return figs

In [ ]:
#| export
def train_flow(model, dataset, n_epochs=100, lr=3e-4, batch_size=2048,
               warp_s=0.5, device='cpu', checkpoint_dir=None, save_every=10,
               eval_every=10, use_wandb=False, steps_per_epoch=None, source_df=None,
               source_scales=None):
    """Train flow matching model on embedding dataset.

    Source: N(0,I) sampled fresh each step.
    Target: embeddings from dataset.
    Loss:   MSE between predicted and true (constant) velocity.
    """
    model = model.to(device)
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                    num_workers=2, pin_memory=(device != 'cpu'), drop_last=True)
    dl_iter = None  # lazy cycle iterator, created if steps_per_epoch is set
    _steps = steps_per_epoch or len(dl)
    if use_wandb:
        import wandb
        wandb.config.update(dict(n_epochs=n_epochs, lr=lr, batch_size=batch_size,
                                 warp_s=warp_s, dim=dataset.embeddings.shape[1]), allow_val_change=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    global_step = 0

    if checkpoint_dir:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.
        if steps_per_epoch:
            if dl_iter is None:
                import itertools; dl_iter = itertools.cycle(dl)
            batches = (next(dl_iter) for _ in range(_steps))
        else:
            batches = dl
        pbar = tqdm(batches, total=_steps, desc=f'Epoch {epoch+1}/{n_epochs}', leave=False)
        for target in pbar:
            target = target.to(device)
            B, D   = target.shape
            source = sample_source((B, D), device=device, source_df=source_df,
                                   source_scales=source_scales,
                                   level_dims=getattr(dataset, 'level_dims', None))
            target = target[:, :source.shape[1]]  # truncate to source dim if dataset has more levels

            t = torch.rand(B, 1, device=device)   # uniform in [0,1]
            if warp_s != 1.0:
                t = warp_time(t, s=warp_s)         # warp for better coverage

            x_t = (1 - t) * source + t * target   # linear interpolation
            v   = target - source                  # constant velocity for straight paths

            optimizer.zero_grad()
            v_pred = model(x_t, t)
            loss   = loss_fn(v_pred, v)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.4f}')
            global_step += 1

        avg_loss = epoch_loss / len(dl)
        print(f'Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}')
        if use_wandb: wandb.log({'train/epoch_loss': avg_loss, 'epoch': epoch+1}, step=global_step)

        if eval_every and (epoch + 1) % eval_every == 0:
            print(f'  --- eval epoch {epoch+1} ---')
            metrics = eval_flow(model, dataset.embeddings, device=device, warp_s=warp_s,
                               source_df=source_df, source_scales=source_scales,
                               level_dims=getattr(dataset, 'level_dims', None))
            if use_wandb:
                import wandb
                log_dict = {f'eval/{k}': v for k, v in metrics.items()}
                if hasattr(dataset, 'level_dims'):
                    figs = plot_level_histograms(model, dataset.embeddings,
                                               dataset.level_dims, device=device, warp_s=warp_s,
                                               source_df=source_df, source_scales=source_scales,
                                               epoch=epoch+1)
                    import matplotlib.pyplot as plt
                    for lname, fig in figs.items():
                        log_dict[f'media/hist_{lname}'] = wandb.Image(fig, caption=f'Epoch {epoch+1}')
                        plt.close(fig)
                log_dict['epoch'] = epoch+1
                wandb.log(log_dict, step=global_step)
            model.train()

        if checkpoint_dir and (epoch + 1) % save_every == 0:
            ckpt = checkpoint_dir / f'flow_epoch{epoch+1:04d}.pt'
            torch.save({'epoch': epoch+1, 'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(), 'loss': avg_loss}, ckpt)
            print(f'  saved {ckpt}')

    return model

In [ ]:
#| export
#| eval: false
import hydra
from omegaconf import DictConfig

@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def train_flow_main(cfg: DictConfig):
    import glob as _glob
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"device = {device}")

    paths = sorted(_glob.glob(os.path.expandvars(os.path.expanduser(cfg.flow.embedding_glob))))
    assert paths, f"No files found matching {cfg.flow.embedding_glob}"
    print(f"Loading {len(paths)} file(s)...")
    source_scales = list(cfg.flow.source_scales) if cfg.flow.get("source_scales") else None
    raw_df = cfg.flow.get("source_df", None)
    source_df = list(raw_df) if hasattr(raw_df, '__iter__') else raw_df
    n_levels = len(source_scales) if source_scales else None
    levels = [f'L{i}' for i in range(n_levels)] if n_levels else None
    if source_df: source_df = source_df[:n_levels]
    dataset = EmbeddingDataset(paths, levels=levels)
    dim = dataset.embeddings.shape[1]
    print(f"  {len(dataset)} samples, dim={dim}, level_dims={dataset.level_dims}")

    model = VelocityNet(input_dim=dim, h_dim=cfg.flow.h_dim, n_layers=cfg.flow.n_layers)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  VelocityNet: {n_params:,} parameters")

    use_wandb = hasattr(cfg, "wandb") and hasattr(cfg.wandb, "flow_project")
    if use_wandb:
        import wandb
        wandb.init(project=cfg.wandb.flow_project, config=dict(cfg.flow))
        wandb.define_metric("epoch")
        wandb.define_metric("*", step_metric="epoch")
        if hasattr(cfg, "tag"): wandb.run.name = f"{cfg.tag}_{wandb.run.name}"

    train_flow(model, dataset,
               n_epochs=cfg.flow.n_epochs, lr=cfg.flow.lr, batch_size=cfg.flow.batch_size,
               warp_s=cfg.flow.warp_s, device=device,
               checkpoint_dir=cfg.flow.checkpoint_dir, save_every=cfg.flow.save_every,
               eval_every=cfg.flow.eval_every, use_wandb=use_wandb,
               steps_per_epoch=cfg.flow.steps_per_epoch,
               source_df=source_df, source_scales=source_scales)

    if use_wandb: wandb.finish()

if __name__ == "__main__":
    train_flow_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()